# Bank agent

* Authenticates user


In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Context

In [2]:
from dataclasses import dataclass

@dataclass
class PinContext:
    user: str = "Marco"
    pin: int = "12345"

In [3]:
from langchain.agents import AgentState

class BankAgentState(AgentState):
    authenticated: bool
    balance: float

/home/emanuele/progetti/LangChain/lca-lc-foundations/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Tools

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_balance(runtime: ToolRuntime) -> str:
    """Check the balanco of your bank account"""
    current_balance = runtime.state.get("balance", 4500.00)    
    return f"""
    Dear Marco, 
    the balance of your bank account amount to {current_balance}
    """

@tool
def transfer_money(to_iban: str, amount: float, description: str, runtime: ToolRuntime) -> str:
    """Send money to a specific IBAN"""
    current_balance = runtime.state.get("balance", 4500.00)    
    new_balance = current_balance - amount

    return Command(update={
        "balance": new_balance,
        "messages": [ToolMessage(
            f"Bank transfer sent to {to_iban} of {amount} $ with description: \"{description}\"",
            tool_call_id=runtime.tool_call_id
        )]
    })

@tool
def deposit_money(amount: float, runtime: ToolRuntime):
    """Deposit money to your bank account balance."""    
    current_balance = runtime.state.get("balance", 4500.00)    
    new_balance = current_balance + amount

    prompt_message = f"{amount}$ added to your back account"

    return Command(update={
        "balance": new_balance,
        "messages": [ToolMessage(
            prompt_message,
            tool_call_id=runtime.tool_call_id
        )]
    })

@tool
def authenticate(user: str, pin: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given user and pin"""
    if user == runtime.context.user and pin == runtime.context.pin:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

### Dynamic tools

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """It allows access to balances and transfers only if the user is authenticated."""
    authenticated = request.state.get("authenticated", False)
    
    if authenticated:
        # if authenticated can check the balance and transfer money
        tools = [check_balance, transfer_money, deposit_money]
    else:
        # if he isn't authenticated can only authenticate
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

### Dynamic Prompt

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a secure virtual banking assistant.
The user has successfully authenticated. Your first step now is to check the user's account balance to welcome them.
"""

unauthenticated_prompt = """You are a virtual banking assistant.
You cannot perform financial transactions until the user authenticates by providing their Username and PIN.
"""

@dynamic_prompt
def bank_dynamic_prompt(request: ModelRequest) -> str:
    """Dynamic prompt for the authenticate state"""
    authenticated = request.state.get("authenticated", False)
    return authenticated_prompt if authenticated else unauthenticated_prompt

### Human-in-the-loop

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_balance, transfer_money, deposit_money],
    checkpointer=InMemorySaver(),
    state_schema=BankAgentState,
    context_schema=PinContext,
    middleware=[
        dynamic_tool_call, 
        bank_dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_balance": False,
                "transfer_money": True,
            })
        ]
    )

## Testing

### Authentication

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "bank_session_1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="My username is Marco and my pin is 12345")]},
    context=PinContext(),
    config=config
)

print(response['messages'][-1].content)

Welcome back, Marco. Your current balance is 4,500.00.

What would you like to do next?
- View recent transactions
- Transfer money to an IBAN
- Deposit funds
- Check other accounts or set up a payment

If you want to transfer, just tell me the IBAN, amount, and a description.


### Back transfer

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Send 500,00$ to IT99L0123456789 for the rent")]},
    context=PinContext(),
    config=config
)

print("STATO INTERROTTO: In attesa di approvazione...")
print(response['__interrupt__'][0].value['action_requests'][0]['args'])

STATO INTERROTTO: In attesa di approvazione...
{'to_iban': 'IT99L0123456789', 'amount': 500, 'description': 'Rent payment'}


### Human approve

In [10]:
from langgraph.types import Command

# Simuliamo l'utente (o l'operatore) che clicca "Approva" sulla UI
response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), 
    config=config
)

print(response["messages"][-1].content)

Done, Marco. I’ve sent 500.00 USD to IT99L0123456789 with the description “Rent payment.”

Your new balance is 4,000.00 USD.

Would you like to view the recent transactions, set up another transfer, or schedule a recurring payment?


In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my current bank balance?")]},
    context=PinContext(),
    config=config
)

print(response['messages'][-1].content)

Your current balance is 4,000.00 USD. Would you like to view recent transactions, make another transfer, or set up a recurring payment?


In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Add 300$ to my bank account")]},
    context=PinContext(),
    config=config
)

print(response['messages'][-1].content)

Great news, Marco! 300.00 USD has been deposited to your account.

Your new balance is 4,300.00 USD.

Would you like to do anything else? You can view transactions, transfer funds, or set up a recurring payment.


In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Add 700$ to my bank account for my next holiday")]},
    context=PinContext(),
    config=config
)

print(response['messages'][-1].content)

Great news, Marco! 700.00 USD has been deposited to your account.

Your new balance is 5,000.00 USD.

Would you like to view recent transactions, transfer funds, or set up a holiday savings goal?


In [14]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my current bank balance?")]},
    context=PinContext(),
    config=config
)

print(response['messages'][-1].content)

Your current balance is 5,000.00 USD. Would you like to view recent transactions, transfer funds, or set up a recurring payment?
